## Библиотеки и константы


In [1]:
import pandas as pd
import numpy as np

import matplotlib
import matplotlib.pyplot as plt
import random
import scipy.stats as stats
import pylab
import struct
import json
import os
from IPython.display import clear_output
from math import factorial
from collections import defaultdict
from itertools import product, combinations, permutations
import inspect
from pathlib import Path

%matplotlib inline

### Базовые значения

In [2]:
stats_map = {
    "HP%": 0,
    "HP": 1, 
    "ATK%": 2,
    "ATK": 3,
    "DEF%": 4,
    "DEF": 5,
    "EM": 6, # Elemental mastery
    "ER": 7, # Energy recharge
    "CRIT_RATE": 8, # Crit Rate
    "CRIT_DMG": 9, # Crit Damage
    "PHYS_DMG": 10,
    "PYRO_DMG": 11,
    "HYDRO_DMG": 12,
    "ANEMO_DMG": 13,
    "ELECTRO_DMG": 14,
    "DENDRO_DMG": 15,
    "CRYO_DMG": 16,
    "GEO_DMG": 17,
    "HEAL_BONUS": 18,
    "None": 100,
}

stat_names = [""] * (max(stats_map.values()) + 1)
for name, stat_id in stats_map.items():
    if stat_id >= 0:
        stat_names[stat_id] = name
        
sub_stats_values_map = {
    stats_map["HP%"]: [408, 466, 525, 583],
    stats_map["HP"]: [20913, 23900, 26988, 29975], 
    stats_map["ATK%"]: [408, 466, 525, 583],
    stats_map["ATK"]: [1362, 1556, 1751, 1945],
    stats_map["DEF%"]: [510, 583, 656, 729],
    stats_map["DEF"]: [1620, 1852, 2083, 2315],
    stats_map["EM"]: [1632, 1865, 2098, 2331],
    stats_map["ER"]: [453, 518, 583, 648],
    stats_map["CRIT_RATE"]: [272, 311, 350, 389],
    stats_map["CRIT_DMG"]: [544, 622, 699, 777]
}

slots = ["FLOWER", "PLUME", "SANDS", "GOBLET", "CIRCLET"]

### Расчет пользы сабстатов

In [3]:
def update_substat_profit(character_name, character, profits):
    char_profit_names = profits.get(character_name)
    isDefault = False
    if char_profit_names is None:
        char_profit_names = profits.get("Default")
        isDefault = True
    character["is_default_substats"] = isDefault

    profit = [0]*10
    for i in range(10):
        profit[i] = char_profit_names[stat_names[i]]
        
    base_stats = character["base_stats"]
    profit[1] = round(profit[0] * sub_stats_values_map[1][-1] * 100 / (base_stats["HP"] * sub_stats_values_map[0][-1]), 2)
    profit[3] = round(profit[2] * sub_stats_values_map[3][-1] * 100 / (base_stats["ATK"] * sub_stats_values_map[2][-1]), 2)
    profit[5] = round(profit[4] * sub_stats_values_map[5][-1] * 100 / (base_stats["DEF"] * sub_stats_values_map[4][-1]), 2)
    character["sub_stat_profit"] = profit

    
def update_all_substat_profit(characters):
    with open("characters/character_profit.json", "r", encoding="utf-8") as file:
        profits = json.load(file)  
    for character_name, character in characters.items():
        score = character["score"]
        if score is None or score < 1:
            update_substat_profit(character_name, character, profits)
    
    
def sort_profit_file():
    with open("characters/character_profit.json", "r", encoding="utf-8") as file:
        profits = json.load(file)
    
    profits = dict(sorted(profits.items()))
    with open("characters/character_profit.json", "w", encoding="utf-8") as file:
        json.dump(
            profits,
            file,
            ensure_ascii=False,
            indent=4
        )


### Парсинг

In [23]:
def init_character(fight_props):
    return {
            "base_stats": {
                "HP": fight_props.get("BASE_HP", 0),
                "ATK": fight_props.get("BASE_ATK", 0),
                "DEF": fight_props.get("BASE_DEF", 0),
                "EM": fight_props.get("EM", 0),
            },
            "score": 0,
            "is_default_substats": True,
            "total_substats": [0]*10,
            "total_rolls": 0,
            "artifacts": {},
            "sub_stat_profit": [0] * 10,
        }

def load_characters(uid):
    characters = {}
    if Path(f"characters/{uid}_parsed.json").exists():
        with open(f"characters/{uid}_parsed.json", "r", encoding="utf-8") as file:
            characters = json.load(file)
    return characters

def parse_characters_from_file(uid):
    with open(f"characters/{uid}.json", "r", encoding="utf-8") as file:
        data = json.load(file)
    
    existed_characters = load_characters(uid)
            
    for new_character in data["characters"]:
        name = new_character["character_name"]
        existed_character = existed_characters.get(name)

        if existed_character is None:
            existed_character = init_character(new_character["fight_props"])
        elif existed_character["score"] > 1:
            continue


        for artifact in new_character["artifacts"]:
            parsed_artifact = {
                "main_stat": artifact["main_stat_prop"],
                "main_stat_value": round(artifact["main_stat_value"] * 100),
                "substats": [0] * 10,
                "first_rolls": {},
                "count_rolls": -4,
                "score": 0,
                "upgrade_chance": {
                    "2_guarant": 0,
                    "3_guarant": 0,
                    "4_guarant": 0,
                    "craft": 0
                }
            }

            for substat in artifact["substats"]:
                stat_name = substat["prop"]
                value = round(substat["value"] * 100)

                # Итоговое значение сабстата конкретного артефакта
                parsed_artifact["substats"][stats_map[stat_name]] = value
                parsed_artifact["first_rolls"][stat_name] = round(substat["rolls"][0] * 100)
                parsed_artifact["count_rolls"] += len(substat["rolls"])

                # Количество роллов этого стата во всём сете
                existed_character["total_substats"][stats_map[stat_name]] += value

            existed_character["artifacts"][artifact["slot"]] = parsed_artifact

        # сортировка слотов артефактов
        artifacts = existed_character["artifacts"]
        existed_character["artifacts"] = {
            slot: artifacts[slot]
            for slot in slots
            if slot in artifacts
        }
        
        existed_characters[name] = existed_character
        
    return existed_characters



def save_characters(characters, uid):
    characters = dict(sorted(characters.items()))
    with open(f"characters/{uid}_parsed.json", "w", encoding="utf-8") as file:
        json.dump(
            characters,
            file,
            ensure_ascii=False,
            indent=4
        )

### Вызовы

In [22]:
characters = parse_characters_from_file(707804862)
update_all_substat_profit(characters)
save_characters(characters, 707804862)

In [7]:
sort_profit_file()

In [29]:
characters = load_characters(707804862)
update_all_substat_profit(characters)
save_characters(characters, 707804862)